# Exercício · Teste A/B do onboarding da Aprendiz

**Disciplina 4 · Testes de Hipótese e Experimentação · Pós-graduação em Ciência de Dados e IA · UNIPDS**

Neste exercício vamos percorrer a anatomia completa de um teste A/B, do planejamento à decisão:

1. **Planejar:** hipóteses, α, poder e MDE, combinados antes de olhar os dados;
2. **Entender o estado atual** do processo com uma EDA do histórico;
3. **Checar o TCL e calcular o tamanho mínimo de amostra** (em Python e no G\*Power);
4. **Analisar o experimento** com as duas lentes da aula: intervalo de confiança e teste de hipótese (multiverso de H0, p-valor);
5. **Escolher o teste certo** para cada métrica: z e t para uma amostra, z para duas proporções e t de Welch (e entender por que os testes de variância não servem aqui);
6. **Separar significância estatística de prática** e tomar a decisão.

Vamos trabalhar com uma **proporção** (conversão) e uma **média** (receita por usuário). **Todos os testes são bicaudais.**

### Como usar este notebook

Cada seção traz um título com o que precisa ser feito. Escreva o código na célula vazia abaixo de cada título (se precisar de mais células, crie) e, depois de rodar, anote em texto a sua leitura do resultado: o que o número diz sobre a pergunta de negócio.

## O contexto

A **Aprendiz** é uma plataforma de cursos online. O usuário se cadastra, ganha **7 dias grátis** para experimentar as aulas e, se gostar, **compra cursos** avulsos. O catálogo tem cursos de R$ 97, R$ 147, R$ 197 e R$ 297, e um usuário pode comprar mais de um.

O time de Produto criou um **onboarding novo, a "trilha guiada"**, que aparece durante os 7 dias grátis: um módulo introdutório com um guia sobre o curso. A aposta é que um usuário que engaja no período grátis compra mais depois. A pergunta de negócio é: **a trilha guiada aumenta as vendas?**

**O desenho do teste**

| Peça | Decisão |
|:---|:---|
| População | Usuários que se cadastram na plataforma  |
| Unidade sorteada | O **usuário**, sorteado no cadastro: 50% controle (onboarding atual), 50% tratamento (trilha guiada) |
| Métrica primária (proporção) | `comprou_30d`: percentual de usuários que fez pelo menos 1 compra nos 30 dias após o cadastro. É atribuido o número 1 se fez a compra, 0 caso contrário |
| Métrica secundária (média) | `receita_30d`: quanto os usuários gastaram em média (R$) nos 30 dias após o cadastro, em cada grupo **0 para quem não comprou** |
| Métrica de diagnóstico | `concluiu_aula1_7d`: 1 se o usuário concluiu a primeira aula dos cursos PAGOS dentro dos 7 dias grátis |

**Por que essas métricas são “limpas”.** As três são calculadas para **todo usuário sorteado**, numa janela fixa a partir do cadastro de cada um. Ninguém fica de fora da conta: quem não comprou entra com 0 na receita. A unidade de análise é a unidade sorteada (o usuário), e nenhuma delas é uma razão.


**Por que 30 dias.** O onboarding age nos 7 dias grátis, mas a compra pode vir depois. Uma janela de 30 dias captura a decisão de compra. O preço disso: cada usuário precisa de 30 dias de acompanhamento antes de entrar na análise. Métricas de negócio são mais lentas que métricas de engajamento.

**Métrica de diagnóstico: conclusão da 1ª aula em 7 dias**

A trilha guiada não vende cursos diretamente. Ela age no período grátis, ajudando o usuário a engajar logo na primeira semana. A aposta do time é uma cadeia de causa e efeito:

> trilha guiada → mais usuários concluem a 1ª aula de algum dos cursos pagos nos 7 dias grátis → mais usuários compram

A conclusão da 1ª aula mede o **elo do meio** dessa cadeia. Ela responde se a trilha fez o que foi desenhada para fazer. Se a conclusão não mudar, o onboarding falhou já no primeiro passo, e qualquer mudança na conversão precisa ser olhada com desconfiança.

**1. Ela confirma se a trilha fez o que prometeu.** O módulo extra existe para despertar interesse pelos cursos pagos. Se ele funciona, mais gente deveria ir olhar esses cursos. A conclusão da primeira aula é o sinal mais direto desse interesse, porque ninguém compra sem antes olhar o curso. Ela mede o passo que liga "vi o módulo" a "comprei".

**2. Ela ajuda a separar onde a trilha falhou ou funcionou.** Combinando com a conversão:

| Conclusão aula | Conversão | Leitura |
|---|---|---|
| sobem | sobe | a trilha funcionou como planejado |
| sobem | não muda | a trilha desperta interesse, mas algo trava na página do curso (preço, descrição, checkout) |
| não mudam | sobe | a trilha vende por outro caminho, e vale investigar qual |
| não mudam | não muda | o módulo não despertou interesse nenhum |

Sem a métrica do meio, um "a conversão não mudou" não diz onde o problema está. Com ela, o time sabe se precisa mexer na trilha ou na página do curso.

**3. Ela responde rápido.** A conversão leva 30 dias para ser medida, e a visita sai em 7. Na primeira semana o time já sabe se o módulo está gerando interesse. Se não estiver, dá para investigar antes de esperar o mês inteiro.

Ela não decide sozinha: mais visitas sem mais compras não pagam a trilha. Ela explica o resultado da métrica que decide, que é a conversão.


## Parte 0 · Preparação

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.proportion import (proportion_confint, proportions_ztest,
                                          confint_proportions_2indep, proportion_effectsize)

ALPHA = 0.05           # nível de significância
PODER = 0.80           # poder desejado
MDE_CONV = 0.015       # 1,5 ponto percentual na conversão
MDE_REC = 3.0          # R$ 3 na receita por usuário

## Parte 1 · As hipóteses e os riscos

Antes de olhar qualquer dado, o time combinou:

| Ingrediente | Valor | Por quê |
|:---|:---|:---|
| Hipóteses (primária) | H0: p_trat = p_ctrl · Ha: p_trat ≠ p_ctrl | Bilateral: se a trilha piorar a conversão, queremos saber |
| Hipóteses (secundária) | H0: μ_trat = μ_ctrl · Ha: μ_trat ≠ μ_ctrl | Bilateral, pelo mesmo motivo |
| α | 5% | Risco aceito de lançar uma trilha que não muda nada (erro tipo I) |
| Poder | 80% | Chance de enxergar o efeito mínimo, se ele existir (β = 20%) |
| MDE da conversão | +1,5 p.p. | Menor ganho que paga o custo de manter a trilha |
| MDE da receita | +R$ 3 por usuário | Menor ganho de receita que paga a trilha |

O que **ainda falta** para calcular o tamanho de amostra é a **variabilidade** das métricas, e precisamos saber se elas se comportam bem o bastante para o TCL. As duas coisas vêm do histórico.

## Parte 2 · O estado atual do processo (EDA)

O arquivo `historico_cadastros.csv` tem os usuários que se cadastraram nas **8 semanas anteriores** ao teste, todos com o onboarding atual e todos com os 30 dias de acompanhamento completos.

### 2.1 Quantos usuários chegam por dia?

### 2.2 A métrica primária: conversão em 30 dias

### 2.3 A métrica secundária: receita por usuário em 30 dias

### 2.4 Diagnóstico: ativação e compra

## Parte 3 · Tamanho mínimo de amostra

### 3.1 Primeiro, o piso do TCL: a regra de Kohavi

### 3.2 Proporção: conversão

### 3.3 Média: receita por usuário

### 3.4 O tamanho do teste e a duração

## Parte 4 · O experimento

O teste rodou por 2 semanas, e a análise foi feita depois que todos os usuários completaram os 30 dias. O arquivo `experimento_onboarding.csv` tem todos os usuários sorteados, com as métricas já medidas.

### 4.1 Checagens antes de analisar

### 4.2 Qual teste usar?

### 4.3 O controle se comporta como o histórico?

## Parte 5 · Métrica primária: conversão

### 5.1 Um primeiro olhar: o IC de cada grupo

### 5.2 Lente 1 · O IC da diferença

### 5.3 Lente 2 · O teste de hipótese no multiverso de H0

### 5.4 Significância estatística e significância prática

## Parte 6 · Métrica secundária: receita por usuário

### 6.1 Resumo dos grupos

### 6.2 O teste t de Welch

### 6.3 Diagnóstico

## Parte 7 · A decisão